# Evaluation: Video-LLaVA-7B on Occludence Benchmark

This notebook evaluates a 4-bit NF4 quantized Video-LLaVA-7B model on RGB and depth-frame versions of the Occludence benchmark.

In [ ]:
!pip install -q transformers==4.47.0 accelerate sentencepiece Pillow tqdm pandas
!pip install -q -U bitsandbytes>=0.46.1

In [ ]:
import os
import json
import re
import gc

import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from collections import defaultdict

from transformers import (
    AutoProcessor,
    LlavaNextVideoForConditionalGeneration,
    BitsAndBytesConfig
)

In [ ]:
if os.path.exists("/kaggle/input"):
    DATASET_DIR = "/kaggle/input/datasets/owaisahmad9870/occcludence"
    BENCHMARK_JSON = os.path.join(DATASET_DIR, "benchmark_qa_297.json")
    FRAMES_BASE_DIR = os.path.join(DATASET_DIR, "benchmark_frames")
    OUTPUT_DIR = "/kaggle/working"
else:
    DATASET_DIR = "/content/dataset"
    BENCHMARK_JSON = os.path.join(DATASET_DIR, "benchmark_qa_297.json")
    FRAMES_BASE_DIR = os.path.join(DATASET_DIR, "verify_frames")
    OUTPUT_DIR = "/content/results"

NUM_FRAMES = 8
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(BENCHMARK_JSON, "r") as file:
    qa_data = json.load(file)

print(f"Loaded {len(qa_data)} QA pairs.")
print(f"Frames directory available: {os.path.exists(FRAMES_BASE_DIR)}")

In [ ]:
def load_frames_for_clip(clip_id, modality="rgb", num_frames=NUM_FRAMES):
    clip_dir = os.path.join(FRAMES_BASE_DIR, clip_id)

    if not os.path.exists(clip_dir):
        return None

    all_files = sorted(os.listdir(clip_dir))

    if modality == "rgb":
        files = [
            file_name for file_name in all_files
            if "_rgb" in file_name and file_name.endswith(".jpg")
        ] or [
            file_name for file_name in all_files
            if "depth" not in file_name and file_name.endswith(".jpg")
        ]
    else:
        files = [
            file_name for file_name in all_files
            if "_depth" in file_name and file_name.endswith(".jpg")
        ] or [
            file_name for file_name in all_files
            if "depth" in file_name and file_name.endswith(".jpg")
        ]

    if not files:
        files = [
            file_name for file_name in all_files
            if file_name.endswith((".jpg", ".png"))
        ]

    if not files:
        return None

    if len(files) > num_frames:
        indices = np.linspace(0, len(files) - 1, num_frames, dtype=int)
        files = [files[index] for index in indices]

    frames = []
    for file_name in files:
        try:
            image = Image.open(
                os.path.join(clip_dir, file_name)
            ).convert("RGB").resize((336, 336))
            frames.append(image)
        except Exception:
            continue

    return frames if frames else None

In [ ]:
def build_prompt(question, options_dict, modality="rgb"):
    options_text = "\n".join(
        f"({key}) {value}"
        for key, value in options_dict.items()
    )

    if modality == "rgb":
        context = (
            "You are watching a sequence of video frames showing a physical "
            "occlusion event on a table."
        )
    else:
        context = (
            "You are watching a sequence of video frames showing a physical "
            "occlusion event on a table. The frames are metric depth maps "
            "where warm colors are closer and cool colors are farther away."
        )

    return (
        f"{context}\n\n"
        f"Question: {question}\n\n"
        f"Options:\n{options_text}\n\n"
        "Answer with ONLY the letter A, B, C, or D. Nothing else.\n"
        "Your answer:"
    )


def parse_answer(raw_text):
    if not raw_text:
        return "INVALID"

    raw_text = raw_text.strip().upper()

    if raw_text in ["A", "B", "C", "D"]:
        return raw_text

    match = re.search(r"\b([A-D])\b", raw_text)
    return match.group(1) if match else raw_text[:1]


def get_options(item):
    return {
        "A": item.get("option_a", item.get("A", "")),
        "B": item.get("option_b", item.get("B", "")),
        "C": item.get("option_c", item.get("C", "")),
        "D": item.get("option_d", item.get("D", ""))
    }


def get_ground_truth(item):
    return str(
        item.get("ground_truth", item.get("answer", ""))
    ).strip().upper()

In [ ]:
MODEL_ID = "llava-hf/LLaVA-NeXT-Video-7B-hf"
print(f"Loading {MODEL_ID}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = LlavaNextVideoForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=bnb_config
)

model.eval()
print("Video-LLaVA-7B loaded successfully.")

In [ ]:
def evaluate_videollava(frames, question, options_dict, modality="rgb"):
    if not frames:
        return "ERROR", ""

    prompt = build_prompt(question, options_dict, modality)
    full_prompt = f"USER: <video>\n{prompt}\nASSISTANT:"

    try:
        video_array = np.stack(
            [np.array(frame) for frame in frames],
            axis=0
        )

        inputs = processor(
            text=full_prompt,
            videos=[video_array],
            return_tensors="pt",
            padding=True
        ).to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False
            )

        generated_ids = output[0][inputs["input_ids"].shape[1]:]
        raw_text = processor.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

        return parse_answer(raw_text), raw_text

    except Exception as error:
        torch.cuda.empty_cache()
        return "ERROR", str(error)

In [ ]:
def run_evaluation(modality="rgb"):
    clip_to_qas = defaultdict(list)

    for item in qa_data:
        clip_to_qas[item["clip_id"]].append(item)

    results = []
    checkpoint_file = f"{OUTPUT_DIR}/videollava_{modality}_checkpoint.csv"

    for clip_id, questions in tqdm(
        clip_to_qas.items(),
        desc=f"Video-LLaVA/{modality}"
    ):
        frames = load_frames_for_clip(clip_id, modality=modality)

        if frames is None:
            continue

        for item in questions:
            prediction, raw_response = evaluate_videollava(
                frames,
                item["question"],
                get_options(item),
                modality
            )

            ground_truth = get_ground_truth(item)

            results.append({
                "clip_id": clip_id,
                "question_id": item["question_id"],
                "task_type": item.get("task_type", ""),
                "scenario": item.get("scenario", ""),
                "module": item.get("module", ""),
                "ground_truth": ground_truth,
                "prediction": prediction,
                "is_correct": prediction == ground_truth,
                "raw_response": raw_response,
                "model": "videollava-7b",
                "modality": modality
            })

        if len(results) % 60 == 0:
            pd.DataFrame(results).to_csv(checkpoint_file, index=False)

        torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)
    output_file = f"{OUTPUT_DIR}/videollava_{modality}_results.csv"
    results_df.to_csv(output_file, index=False)

    accuracy = results_df["is_correct"].mean() * 100
    print(
        f"Video-LLaVA {modality} accuracy: {accuracy:.2f}% | "
        f"Saved to {output_file}"
    )

    return results_df


df_rgb = run_evaluation("rgb")
df_depth = run_evaluation("depth")